# C. elegans: Load, Preprocess, and Window

This dataset-level notebook creates the immutable recording and temporal-window checkpoints shared by every method lane.
Confirm the raw-data path, array keys, sampling rate, behavior mapping, valid ranges, and gaps in the named dataset config before running.
Progress: long stages print checkpoint-aware timing; iterative stages also display live progress bars. Reused checkpoints are reported explicitly.


In [ ]:
import sys
from pathlib import Path

try:
    _REPOSITORY_ROOT = next(
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "notebooks" / "_shared.py").is_file()
    )
except StopIteration as error:
    raise RuntimeError(
        "Could not locate the effectome repository from the notebook working directory"
    ) from error
sys.path[:0] = [
    path
    for path in (str(_REPOSITORY_ROOT / "src"), str(_REPOSITORY_ROOT))
    if path not in sys.path
]


In [ ]:
from omegaconf import OmegaConf

from effectome.data_module import PreprocessConfig, WindowConfig
from notebooks._shared import (
    PROJECT_ROOT,
    make_run,
    print_stage_status,
    run_bundle_net_reference_preprocessing,
    run_preprocessing,
    stage_artifact_path,
)

DATASET_ID = "c_elegans"
DATASET_OUTPUT_ID = "bundle-net-c-elegans"
RECORDING_ID = "bundle-net-worm-0"
RUN_ID = "reference"
METHOD = "shared"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "analysis" / DATASET_OUTPUT_ID / RECORDING_ID
DATA_CONFIG_PATH = PROJECT_ROOT / "conf" / "data" / "c_elegans.yaml"
WINDOW_CONFIG_PATH = PROJECT_ROOT / "conf" / "windowing" / "primary_physical.yaml"
FORCE = False

DATA_CFG = OmegaConf.to_container(OmegaConf.load(DATA_CONFIG_PATH), resolve=True)
if not isinstance(DATA_CFG, dict):
    raise TypeError(f"Expected a mapping in {DATA_CONFIG_PATH}")
DATA_CFG["path"] = str((PROJECT_ROOT / str(DATA_CFG["path"])).resolve())
WINDOWING_CFG = OmegaConf.to_container(OmegaConf.load(WINDOW_CONFIG_PATH), resolve=True)
if not isinstance(WINDOWING_CFG, dict):
    raise TypeError(f"Expected a mapping in {WINDOW_CONFIG_PATH}")
if str(DATA_CFG.get("dataset_id", "")) != DATASET_OUTPUT_ID:
    raise ValueError(
        f"Notebook output dataset id {DATASET_OUTPUT_ID} does not match config dataset id "
        f"{DATA_CFG.get('dataset_id')}"
    )
if str(DATA_CFG.get("recording_id", "")) != RECORDING_ID:
    raise ValueError(
        f"Notebook output recording id {RECORDING_ID} does not match config recording id "
        f"{DATA_CFG.get('recording_id')}"
    )

PREPROCESS_CFG = PreprocessConfig(
    detrend=False,
    zscore=False,
    deconvolve=False,
    smooth_window=0,
    drop_low_variance=0.0,
)

BUNDLE_NET_REFERENCE_CFG = PreprocessConfig(
    **DATA_CFG["bundle_net_reference"]["preprocessing"]
)
BUNDLE_NET_TARGET_LENGTH = int(
    DATA_CFG["bundle_net_reference"]["paired_window_length"]
)
BUNDLE_NET_EXCLUDE_NEURONS = list(
    DATA_CFG["bundle_net_reference"]["exclude_neuron_names"]
)
BUNDLE_NET_EXCLUDE_NAME_SOURCE = str(
    DATA_CFG["bundle_net_reference"]["exclude_neuron_name_source"]
)

WINDOW_CFG = WindowConfig(**WINDOWING_CFG)
EXPECTED_BEHAVIOR_KEYS = ["motif"]

run = make_run(OUTPUT_ROOT, RUN_ID, METHOD)
print_stage_status(run)


In [ ]:
recording, windows = run_preprocessing(
    run,
    data_cfg=DATA_CFG,
    preprocess_cfg=PREPROCESS_CFG,
    window_cfg=WINDOW_CFG,
    force=FORCE,
)
missing_behavior = sorted(set(EXPECTED_BEHAVIOR_KEYS) - set(recording.behavior))
if missing_behavior:
    raise KeyError(f"Dataset is missing configured behavior keys: {missing_behavior}")

print(f"recording: {recording.n_neurons} neurons x {recording.n_timepoints} samples")
print(f"windows: {windows.n_windows}; shape={windows.segments.shape}")
timing = windows.metadata["temporal_contract"]
print(
    "duration-first timing: "
    f"history={timing['history_length']} frames/{timing['history']['resolved_seconds']:.3f}s, "
    f"target={timing['target_length']} frames/{timing['target']['resolved_seconds']:.3f}s, "
    f"stride={timing['stride_length']} frames/{timing['stride']['resolved_seconds']:.3f}s, "
    f"adjacent-context overlap={1.0 - timing['stride_length'] / timing['history_length']:.1%}"
)
print(f"recording artifact: {stage_artifact_path(run, 'recording')}")
print(f"windows artifact: {stage_artifact_path(run, 'windows')}")
print_stage_status(run)


In [ ]:
bundle_reference, bundle_pairs = run_bundle_net_reference_preprocessing(
    run,
    preprocess_cfg=BUNDLE_NET_REFERENCE_CFG,
    exclude_neuron_names=BUNDLE_NET_EXCLUDE_NEURONS,
    exclude_neuron_name_source=BUNDLE_NET_EXCLUDE_NAME_SOURCE,
    behavior_key="motif",
    target_length=BUNDLE_NET_TARGET_LENGTH,
    force=FORCE,
)
print(
    "Bundle-Net reference: "
    f"{bundle_reference.n_neurons} neurons x {bundle_reference.n_timepoints} samples; "
    f"paired windows={bundle_pairs.x_t.shape[0]}"
)
print(
    "reference recording artifact: "
    f"{stage_artifact_path(run, 'bundle_net_reference_recording')}"
)
print(f"training-pair artifact: {stage_artifact_path(run, 'bundle_net_training_pairs')}")
print_stage_status(run)
